# Long sim-VLA run — analysis

Tables, plots and head-to-head episode GIFs for a
`test/system/long_vla_sim.sh` run directory.

**Usage** — point `RUN_DIR` at a run (defaults to the latest under
`$SO101_OUTPUT_DIR/vla_sim_long/`, or set the `LONG_VLA_RUN_DIR`
environment variable), then *Run All*. Every section skips gracefully
when its inputs are missing (e.g. `--only` runs or a smoke-sized dir).

Needs the repo venv kernel (`venv/bin/pip install notebook` or open in
VSCode and select `venv/bin/python`).

In [ ]:
# --- setup: repo imports + run-dir resolution -------------------------
import json, os, sys
from pathlib import Path

_here = Path.cwd()
REPO = _here.parent if _here.name == "notebooks" else _here
for p in (str(REPO), str(REPO / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sim_datagen.report import MODES, POLICIES, TASKS, collect

_env = os.environ.get("LONG_VLA_RUN_DIR")
if _env:
    RUN_DIR = Path(_env)
else:
    _root = Path(os.environ.get("SO101_OUTPUT_DIR", REPO / "outputs")) / "vla_sim_long"
    _runs = sorted(_root.glob("*")) if _root.is_dir() else []
    RUN_DIR = _runs[-1] if _runs else None
assert RUN_DIR and RUN_DIR.is_dir(), (
    "No run directory found — set LONG_VLA_RUN_DIR or RUN_DIR by hand")
print("Analysing:", RUN_DIR)
data = collect(RUN_DIR)
cells_data = data["cells"]
print(f"{len(cells_data)} policy cells, "
      f"{len(data['oracle_gate'])} gate files, "
      f"{len(data['collection'])} collection stats")

In [ ]:
# --- headline tables ---------------------------------------------------
def eval_frame():
    rows = []
    for c in cells_data:
        ev = c.get("eval") or {}
        sel = c.get("selected") or {}
        rows.append({
            "mode": c["mode"], "task": c["task"], "policy": c["policy"],
            "success": ev.get("success_rate"),
            "place_err_mm": ev.get("place_err_mm_mean"),
            "place_err_p95": ev.get("place_err_mm_p95"),
            "selected_step": sel.get("step"),
            "val_success": sel.get("val_success"),
        })
    return pd.DataFrame(rows)

df = eval_frame()
if df.empty:
    print("No cells yet.")
else:
    display(df.pivot_table(index=["mode", "task"], columns="policy",
                           values="success").round(2))
    display(df.set_index(["mode", "task", "policy"]).round(2))

if data["oracle_gate"]:
    display(pd.DataFrame([
        {"cell": k, **{kk: v.get(kk) for kk in
         ("task", "oracle_mode", "oracle_success_rate", "episode_attempts")}}
        for k, v in data["oracle_gate"].items() if v]).round(2))

In [ ]:
# --- success-rate bars by mode/task ------------------------------------
plot_df = df.dropna(subset=["success"]) if not df.empty else df
if plot_df.empty:
    print("No eval results to plot.")
else:
    groups = sorted({(r["mode"], r["task"]) for _, r in plot_df.iterrows()})
    x = np.arange(len(groups))
    width = 0.8 / max(len(POLICIES), 1)
    fig, ax = plt.subplots(figsize=(1.8 * len(groups) + 3, 3.5))
    for i, pol in enumerate(POLICIES):
        vals = []
        for g in groups:
            m = plot_df[(plot_df["mode"] == g[0]) & (plot_df["task"] == g[1])
                        & (plot_df["policy"] == pol)]
            vals.append(float(m["success"].iloc[0]) if len(m) else np.nan)
        ax.bar(x + (i - 1) * width, vals, width, label=pol)
    ax.set_xticks(x, [f"{m}\n{t}" for m, t in groups])
    ax.set_ylim(0, 1.05); ax.set_ylabel("success rate"); ax.legend()
    ax.set_title("Final evaluation (selected checkpoints)")
    plt.tight_layout(); plt.show()

In [ ]:
# --- per-seed success heatmap ------------------------------------------
for mode in MODES:
    for task in TASKS:
        rows, seeds = [], None
        for pol in POLICIES:
            cell = next((c for c in cells_data
                         if (c["mode"], c["task"], c["policy"]) == (mode, task, pol)
                         and c.get("eval")), None)
            if not cell:
                continue
            eps = cell["eval"].get("episodes") or []
            if not eps:
                continue
            s = [e.get("scenario_seed") for e in eps]
            if seeds is None:
                seeds = s
            rows.append((pol, [1.0 if e.get("success") else 0.0 for e in eps]))
        if not rows or seeds is None:
            continue
        fig, ax = plt.subplots(figsize=(max(6, 0.28 * len(seeds)), 0.6 * len(rows) + 1))
        ax.imshow([r[1] for r in rows], vmin=0, vmax=1, aspect="auto", cmap="RdYlGn")
        ax.set_yticks(range(len(rows)), [r[0] for r in rows])
        ax.set_xticks(range(len(seeds)), seeds, rotation=90, fontsize=7)
        ax.set_title(f"per-seed success — {mode}/{task}")
        plt.tight_layout(); plt.show()

In [ ]:
# --- validation curves (success vs checkpoint step) --------------------
fig, ax = plt.subplots(figsize=(7, 4))
plotted = False
for c in cells_data:
    val = c.get("val") or {}
    pts = sorted((int(k.split("_", 1)[1]), v["success_rate"])
                 for k, v in val.items() if v)
    if len(pts) < 1:
        continue
    plotted = True
    ax.plot([p[0] for p in pts], [p[1] for p in pts], marker="o",
            label=f"{c['mode']}/{c['task']}/{c['policy']}")
if plotted:
    ax.set_xlabel("checkpoint step"); ax.set_ylabel("val success rate")
    ax.set_ylim(-0.05, 1.05); ax.legend(fontsize=8)
    ax.set_title("Checkpoint validation (VAL seeds)")
    plt.tight_layout(); plt.show()
else:
    print("No validation results.")
    plt.close(fig)

In [ ]:
# --- place-error distributions -----------------------------------------
fig, ax = plt.subplots(figsize=(7, 4))
plotted = False
for c in cells_data:
    ev = c.get("eval") or {}
    errs = [e["place_err_mm"] for e in (ev.get("episodes") or [])
            if "place_err_mm" in e]
    if not errs:
        continue
    plotted = True
    ax.hist(errs, bins=20, alpha=0.5,
            label=f"{c['mode']}/{c['task']}/{c['policy']}")
if plotted:
    ax.set_xlabel("place error (mm)"); ax.set_ylabel("episodes")
    ax.legend(fontsize=8); ax.set_title("Final-eval place error")
    plt.tight_layout(); plt.show()
else:
    print("No per-episode place errors.")
    plt.close(fig)

In [ ]:
# --- head-to-head episode GIF ------------------------------------------
# Composes each policy's saved eval video of the SAME scenario side by
# side (resampled to a common clock, labelled) and writes the GIF into
# the run directory. Auto-picks the first (mode, task, seed) that has at
# least one video; override GIF_MODE / GIF_TASK / GIF_SEED to choose.
import imageio.v2 as imageio
from PIL import Image, ImageDraw

GIF_MODE = GIF_TASK = GIF_SEED = None  # override here if wanted
GIF_FPS, GIF_FRAMES = 15, 150

def _videos_for(mode, task):
    out = {}
    for pol in POLICIES:
        vdir = RUN_DIR / mode / task / pol / "eval" / "videos"
        for f in sorted(vdir.glob("ep_seed*.mp4")) if vdir.is_dir() else []:
            seed = f.stem.replace("ep_seed", "").split("_")[0]
            out.setdefault(seed, {})[pol] = f
    return out

picked = None
for mode in ([GIF_MODE] if GIF_MODE else MODES):
    for task in ([GIF_TASK] if GIF_TASK else TASKS):
        vids = _videos_for(mode, task)
        if GIF_SEED is not None:
            vids = {k: v for k, v in vids.items() if k == str(GIF_SEED)}
        if vids:
            seed = sorted(vids)[0]
            picked = (mode, task, seed, vids[seed])
            break
    if picked:
        break

if not picked:
    print("No eval videos found (run the final eval with --video-dir).")
else:
    mode, task, seed, vids = picked
    print(f"Composing {mode}/{task} seed {seed}: {sorted(vids)}")
    clips = {}
    for pol, path in vids.items():
        frames = imageio.mimread(path, memtest=False)
        idx = np.linspace(0, len(frames) - 1, GIF_FRAMES).astype(int)
        clips[pol] = [frames[i] for i in idx]
    h = min(c[0].shape[0] for c in clips.values())
    comp = []
    for k in range(GIF_FRAMES):
        panels = []
        for pol in sorted(clips):
            img = Image.fromarray(clips[pol][k])
            img = img.resize((int(img.width * h / img.height), h))
            ImageDraw.Draw(img).text((6, 6), pol, fill=(255, 255, 0))
            panels.append(np.asarray(img))
        comp.append(np.hstack(panels))
    out_gif = RUN_DIR / f"head2head_{mode}_{task}_seed{seed}.gif"
    imageio.mimsave(out_gif, comp, fps=GIF_FPS)
    print("wrote", out_gif)
    from IPython.display import Image as IPyImage, display
    display(IPyImage(filename=str(out_gif)))

---
The tables above mirror `results.md` (regenerate it with
`python -m sim_datagen.report <run_dir>`); the plots and GIFs feed the
`sim_training` paper's experiments section.